# Construire des applications de génération d’images (OpenAI)

Les LLM ne se limitent pas à la génération de texte. Vous pouvez également générer des images à partir de descriptions textuelles. Les images en tant que modalité sont utiles dans la MedTech, l’architecture, le tourisme, le développement de jeux, le marketing, et plus encore. Dans cette leçon, nous travaillons avec les modèles **GPT Image** actuels sur la plateforme OpenAI.

## Objectifs d’apprentissage

À la fin de cette leçon, vous serez capable de :

- Expliquer ce qu’est la génération d’images et où elle est utile.
- Comprendre la famille de modèles `gpt-image` et en quoi elle diffère des anciens modèles DALL·E.
- Construire une application de génération d’images et éditer des images.

## Qu’est-ce que la génération d’images ?

Les modèles de génération d’images créent des images à partir d’une invite textuelle. Les modèles modernes comme `gpt-image` apprennent la relation entre le texte et les images pendant l’entraînement, puis transforment itérativement un bruit aléatoire en une image correspondant à votre invite.

Deux familles bien connues de modèles d’images sont :

- **`gpt-image` (OpenAI)** — la génération actuelle utilisée dans cette leçon. Il supporte la génération texte-vers-image et l’édition d’images (inpainting avec un masque).
- **Midjourney** — un modèle tiers populaire avec son propre service et un workflow basé sur Discord.

> Les anciens modèles d’image OpenAI — **DALL·E 2** et **DALL·E 3** — sont hérités. DALL·E 3 n’est plus disponible pour de nouveaux déploiements, et des fonctionnalités comme `create_variation` existaient uniquement dans DALL·E 2. Utilisez les modèles `gpt-image` pour de nouvelles applications.

> **Important :** Les modèles `gpt-image` renvoient l’image générée en **base64** (`b64_json`), pas sous forme d’URL. Votre code décode la chaîne base64 en octets et la sauvegarde — il n’y a pas d’URL d’image à télécharger.


## Construire votre première application de génération d'images

Que faut-il pour construire une application de génération d'images ? Vous avez besoin des bibliothèques suivantes :

- **python-dotenv**, il est fortement recommandé d'utiliser cette bibliothèque pour garder vos secrets dans un fichier *.env* à l'écart du code.
- **openai**, cette bibliothèque est celle que vous utiliserez pour interagir avec l'API OpenAI.
- **pillow**, pour travailler avec des images en Python.


1. Créez un fichier *.env* avec le contenu suivant :

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Rassemblez les bibliothèques ci-dessus dans un fichier appelé *requirements.txt* comme suit :

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ensuite, créez un environnement virtuel et installez les bibliothèques :


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pour Windows, utilisez les commandes suivantes pour créer et activer votre environnement virtuel :

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Ajoutez le code suivant dans un fichier appelé *app.py* :

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Créez un objet OpenAI (lit OPENAI_API_KEY depuis votre fichier .env)
    client = openai.OpenAI()


    try:
        # Créez une image en utilisant l'API de génération d'images
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Entrez votre texte d'invite ici
            size='1024x1024',
            n=1
        )
        # Définissez le répertoire pour l'image stockée
        image_dir = os.path.join(os.curdir, 'images')

        # Si le répertoire n'existe pas, créez-le
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialisez le chemin de l'image (notez que le type de fichier doit être png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Les modèles gpt-image renvoient l'image en base64 (b64_json), pas une URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Affichez l'image dans le visualiseur d'images par défaut
        image = Image.open(image_path)
        image.show()

    # attraper les exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Expliquons ce code :

- D'abord, nous importons les bibliothèques dont nous avons besoin, y compris la bibliothèque OpenAI, la bibliothèque dotenv, le module base64, et la bibliothèque Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Ensuite, nous créons le client, qui lit la clé API depuis votre fichier ``.env``.

    ```python
    # Créer un objet OpenAI
    client = openai.OpenAI()
    ```

- Ensuite, nous générons l'image :

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Entrez votre texte d'invite ici
        size='1024x1024',
        n=1
    )
    ```

    Les modèles `gpt-image` retournent l'image sous forme de chaîne **base64** dans `data[0].b64_json`. Nous la décodons en octets et l'écrivons dans un fichier — il n'y a pas d'URL à télécharger.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Enfin, nous ouvrons l'image et utilisons le visualiseur d'image standard pour l'afficher :

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Plus de détails sur la génération de l'image

Regardons les paramètres de `images.generate` :

- **model**, est le modèle d'image à utiliser (par exemple `gpt-image-1`).
- **prompt**, est la description textuelle utilisée pour générer l'image. Ici, c'est "Lapin sur un cheval, tenant une sucette, sur une prairie brumeuse où poussent des jonquilles".
- **size**, est la taille de l'image générée (par exemple `1024x1024`, `1536x1024`, `1024x1536`, ou `"auto"`).
- **n**, est le nombre d'images générées. Ici, nous en générons une.

> Les modèles d'image ne prennent pas de paramètre `temperature` — c'est un contrôle pour la génération de texte. Pour obtenir de la variété, appelez l'API à nouveau ; pour réduire la variété, rendez votre description plus spécifique.

## Capacités supplémentaires de la génération d'images

Vous avez vu comment générer une image avec quelques lignes de Python. Les modèles `gpt-image` peuvent aussi **éditer** une image existante. En fournissant une image, un **masque** optionnel (qui marque la zone à modifier) et une description, vous pouvez modifier une partie de l'image — par exemple, ajouter un chapeau à notre lapin.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# les modifications sont également renvoyées sous forme de base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

L'image de base contient juste le lapin ; l'image finale a le chapeau sur le lapin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
